# Day 3 — Lazy Evaluation & the Catalyst Optimizer

**What you will learn:**
- The difference between transformations and actions
- How Spark builds a DAG before executing anything
- How the Catalyst optimizer rewrites your plan
- Reading `explain()` output at all verbosity levels
- Adaptive Query Execution (AQE)

**Exam relevance:** Architecture (20%) + Tuning (10%) — lazy evaluation is one of the most-tested concepts.

## 1. Transformations vs Actions

| Transformations (lazy — build the plan) | Actions (trigger execution) |
|---|---|
| `select()` | `show()` |
| `filter()` / `where()` | `count()` |
| `withColumn()` | `collect()` |
| `groupBy()` | `take(n)` |
| `join()` | `first()` |
| `orderBy()` | `write()` |
| `union()` | `toPandas()` |
| `distinct()` | `foreach()` |
| `repartition()` | `saveAsTextFile()` |

> **Rule:** If it returns a DataFrame → transformation (lazy).  
> If it returns a Python value or writes data → action (triggers execution).

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum

data = [(i, f"dept_{i % 3}", i * 15) for i in range(1, 13)]
df = spark.createDataFrame(data, ["id", "dept", "revenue"])

# All transformations — nothing runs here
step1 = df.filter(col("revenue") > 50)
step2 = step1.withColumn("revenue_k", col("revenue") / 1000)
step3 = step2.groupBy("dept").agg(spark_sum("revenue_k").alias("total_k"))

print("Plan built — zero data processed yet")

# THIS is when Spark runs the full plan in one optimised pass
step3.show()

## 2. The DAG (Directed Acyclic Graph)

When you chain transformations, Spark builds a DAG — a graph of dependencies between partitions.

```
df (scan)  ──►  filter  ──►  withColumn  ──►  groupBy  ──►  show()
                                                   ↑
                                               SHUFFLE
                                             (stage boundary)
```

The DAG is **acyclic** — no loops. Spark walks the DAG backwards from the action to figure out what needs to be computed.  
This also enables **fault recovery**: re-run only the failed branch of the DAG.

## 3. Reading explain() Output

`explain()` shows you what Spark *actually* plans to do — not what you wrote.

| Mode | What you see |
|---|---|
| `explain()` or `explain(False)` | Physical plan only |
| `explain(True)` | Parsed → Analysed → Optimised → Physical |
| `explain("formatted")` | Clean multi-section output (Spark 3+) |
| `explain("cost")` | Physical plan + cost estimates |
| `explain("codegen")` | Generated Java code |


In [ ]:
# Physical plan only
print("=== Physical Plan ===")
step3.explain()

In [ ]:
# Full plans: Parsed → Analysed → Optimised → Physical
print("=== All Plans ===")
step3.explain(True)

In [ ]:
# Formatted — easiest to read (Spark 3+)
step3.explain("formatted")

## 4. The Catalyst Optimizer

Catalyst is Spark's query optimizer. It takes your logical plan and rewrites it to be more efficient *before* running anything.

```
Your Python code
       │
       ▼
  Unresolved Logical Plan   (parse)
       │
       ▼
  Resolved Logical Plan     (bind column names to schema)
       │
       ▼
  Optimised Logical Plan    ◄── Catalyst rewrites here
       │  (predicate pushdown, constant folding, column pruning)
       ▼
  Physical Plans            (pick best join strategy, etc.)
       │
       ▼
  Selected Physical Plan
       │
       ▼
  Tungsten Execution        (whole-stage codegen, off-heap memory)
```

**Key optimisations Catalyst applies automatically:**
- **Predicate pushdown** — moves filters as early as possible (closer to the data source)
- **Column pruning** — reads only the columns you actually use
- **Constant folding** — evaluates constant expressions at plan time
- **Join reordering** — puts smaller tables first

In [ ]:
# Catalyst predicate pushdown in action
# Even though we filter AFTER select, Catalyst pushes the filter down
df2 = df.select("id", "dept", "revenue").filter(col("revenue") > 100)
df2.explain("formatted")
# Notice: in the physical plan, the Filter appears BEFORE (or inside) the Project (select)

## 5. Adaptive Query Execution (AQE)

AQE (Spark 3.0+) takes optimization further by **re-optimizing at runtime** based on actual data statistics.

What AQE does automatically:
| Feature | What it fixes |
|---|---|
| **Coalesce shuffle partitions** | Merges too-small post-shuffle partitions |
| **Switch join strategies** | Converts sort-merge join → broadcast join if one side is small |
| **Skew join optimisation** | Splits skewed partitions to avoid one slow task |

> **Exam tip:** AQE is enabled by default in Spark 3.2+. Setting `spark.sql.adaptive.enabled = true` enables it explicitly.

In [ ]:
# Check and enable AQE
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

# Enable explicitly
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Default shuffle partitions (200 by default — often too many for small data)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# AQE will coalesce these down automatically based on actual data size
spark.conf.set("spark.sql.shuffle.partitions", "200")  # reset to default

## 6. Common Mistakes

| Mistake | Why it's wrong | Correct thinking |
|---|---|---|
| Calling `.collect()` on large DataFrames | Brings ALL data to Driver — OOM | Use `.show()` or write to storage |
| Chaining actions in a loop | Each action is a full job | Materialise once, reuse |
| Expecting immediate execution after `.filter()` | Transformations are lazy | Nothing runs until an action |
| Ignoring `explain()` | Optimizer may be doing something unexpected | Always check the physical plan |


---

## Day 3 Checklist

- [ ] Can classify any operation as transformation or action without hesitation
- [ ] Understand: transformations build the DAG, actions execute it
- [ ] Can read a physical plan from `explain()` and identify key nodes
- [ ] Know the 4 phases of Catalyst (Unresolved → Resolved → Optimised → Physical)
- [ ] Know 3 optimisations Catalyst applies automatically
- [ ] Can explain what AQE is and what it solves
- [ ] Know why `.collect()` on large data is dangerous

**Next:** Day 4 — Column Operations (col, lit, when/otherwise, cast)